# 05. Agentic RAG -- LangGraph로 직접 구축

Retrieval-Augmented Generation(RAG)을 세 가지 방법으로 구현합니다:
1. **LangChain RAG Agent** — `create_agent` + `@tool`로 에이전트가 자율적으로 검색
2. **LangChain RAG Chain** — `@dynamic_prompt` 미들웨어로 단일 LLM 호출 RAG
3. **LangGraph StateGraph 커스텀 RAG** — 문서 관련성 평가, 쿼리 리라이트, 조건부 라우팅 등 심화 패턴

각 방식의 장단점을 비교하고, 상황에 맞는 RAG 전략을 선택할 수 있도록 합니다.

## 학습 목표

- RAG 파이프라인(인덱싱 -> 검색 -> 생성)의 전체 구조를 이해한다
- `RecursiveCharacterTextSplitter`로 문서를 청킹한다
- `InMemoryVectorStore`로 벡터 스토어를 구축한다
- LangChain `create_agent` + `@tool`로 RAG Agent를 구현한다
- `@dynamic_prompt` 미들웨어로 RAG Chain(단일 LLM 호출)을 구현한다
- LangGraph `StateGraph`로 커스텀 RAG 에이전트를 구축한다
- `GradeDocuments` 구조화 출력으로 문서 관련성을 평가한다
- 쿼리 리라이트와 조건부 라우팅을 구현한다

## 5.1 환경 설정

LLM과 임베딩 모델을 초기화합니다.
- **LLM**: `gpt-5.4-mini` — 질의 응답 및 문서 평가에 사용
- **Embeddings**: `text-embedding-3-small` — 문서/쿼리를 벡터로 변환하여 유사도 검색에 사용

In [8]:
# .env 파일에서 API 키(OPENAI_API_KEY 등)를 로드합니다.
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LLM: 질의 응답, 문서 관련성 평가, 쿼리 리라이트에 사용
llm = ChatOpenAI(model="gpt-5.4-mini")
# 임베딩 모델: 문서와 쿼리를 벡터 공간에 매핑하여 유사도 검색 수행
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print("환경 준비 완료.")

환경 준비 완료.


In [9]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


Langfuse tracing ON — 


## 5.2 RAG 개요

RAG(Retrieval-Augmented Generation)는 외부 지식을 검색하여 LLM 응답의 정확도를 높이는 패턴입니다. LLM은 두 가지 핵심 제약을 가집니다:
- **유한한 컨텍스트**: 전체 코퍼스를 한 번에 처리할 수 없음
- **정적 지식**: 학습 데이터가 시간이 지나면 구식이 됨

RAG는 쿼리 시점에 관련 외부 정보를 가져와 이 제약을 극복합니다.

### 파이프라인: 인덱싱 -> 검색 -> 생성

```
[문서] -> Text Splitter -> [청크] -> Embeddings -> [벡터 스토어]
                                                      |
[질문] -> Embedding -> similarity_search -> [관련 청크] -> LLM -> [답변]
```

### 5가지 핵심 구성 요소

| 구성 요소 | 역할 |
|---|---|
| **Document Loader** | 다양한 소스(PDF, 웹, DB 등)에서 문서를 로드 |
| **Text Splitter** | 문서를 검색에 적합한 크기의 청크로 분할 |
| **Embedding Model** | 텍스트를 고차원 벡터로 변환 |
| **Vector Store** | 벡터를 저장하고 유사도 검색을 수행 |
| **Retriever** | 쿼리와 가장 관련 있는 문서를 반환 |

## 5.3 문서 로딩 & 청킹

### 문서 로더 (Document Loaders)
문서 로더는 다양한 소스에서 원시 콘텐츠를 읽어 `page_content`와 `metadata` 필드를 가진 `Document` 객체로 반환합니다.

| 로더 | 소스 | 패키지 |
|---|---|---|
| `PyPDFLoader` | PDF 파일 | `pypdf` |
| `TextLoader` | 텍스트 파일 | 내장 |
| `CSVLoader` | CSV 파일 | 내장 |
| `WebBaseLoader` | 웹 페이지 | `beautifulsoup4` |
| `DirectoryLoader` | 디렉토리 내 파일들 | 내장 |

### 텍스트 분할 (Text Splitting)
`RecursiveCharacterTextSplitter`는 `\n\n` -> `\n` -> ` ` -> `""` 순으로 재귀적으로 분할하여 의미적 연관성을 유지합니다. 가장 범용적인 분할기로 권장됩니다.

| 파라미터 | 설명 | 권장값 |
|---|---|---|
| `chunk_size` | 청크 최대 문자 수 | 500-2000 (작으면 정밀 검색, 크면 맥락 보존) |
| `chunk_overlap` | 인접 청크 공유 문자 수 | chunk_size의 10-20% (경계 정보 손실 방지) |

### 기타 분할기

| 분할기 | 적합한 경우 |
|---|---|
| `MarkdownHeaderTextSplitter` | 마크다운 문서 |
| `HTMLHeaderTextSplitter` | HTML 문서 |
| `TokenTextSplitter` | 토큰 예산 기반 분할 |
| `CodeTextSplitter` | 소스 코드 (언어 인식) |

In [22]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
# pdf File loader 실습을 위해 추가
from langchain_community.document_loaders import PyPDFLoader

file_path = "../AI_Brief_Mar_260303.pdf"
loader = PyPDFLoader(file_path)
docs = loader.load()
docs[0].page_content[:500]  # 첫 페이지의 처음 500자 출력


# 샘플 문서 생성 — 실제 환경에서는 PDF, 웹 등에서 로드합니다.
# Document 객체는 page_content(본문)와 metadata(출처 등)로 구성됩니다.
raw_docs = [
    Document(page_content="LangGraph는 LLM으로 상태 기반 멀티 액터 "
        "애플리케이션을 구축하기 위한 프레임워크입니다.",
        metadata={"source": "langgraph-docs"}),
    Document(page_content="에이전트는 도구를 사용하여 외부 시스템과 "
        "상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다.",
        metadata={"source": "agent-guide"}),
]
print(f"문서 {len(raw_docs)}개 로드됨.")

문서 2개 로드됨.


In [24]:
print(docs[1].page_content)  # 두 번째 페이지의 처음 500자 출력

Ⅰ 년 월호2026 1 Ⅰ
월간 모델현황AI∙ 년 월 월간 모델 현황 2026 2 AI 1
정책･법제
∙앤트로픽 국방부 가드레일 갈등                                                               · (DoW) AI 3
∙미국 캘리포니아주 상원 안전 표준 수립 법안 가결, AI 4
∙영국 정부 년 국제 안전 보고서 발간, 2026 AI 5
∙미국 노동부 리터러시 프레임워크 발표, AI 6
핵심 산업 분야의 도입 현황과 과제 분석OECD, EU AI ∙ 7
∙인도 정부 글로벌 정상회의 임팩트 서밋 개최, AI ‘AI ’ 8
기업･산업
∙마이크로소프트 차세대 추론 칩 마이아 공개, AI ‘ 200’ 10
∙중국 기업들의 최신 모델 출시 동향AI AI 11
∙오픈 연구 논문 작성과 협업 촉진하는 워크스페이스 프리즘 출시AI, AI ‘ ’ 12
∙앤트로픽 에이전트 협업 기능 강화한 출시      , ‘Claude Opus 4.6’ 13  
일본 구마모토현에서 나노급 반도체 공장 건설 계획TSMC, 3∙ 14
∙오픈 에이전트형 코딩 모델 공개AI, AI ‘GPT-5.3-Codex’ 15
∙마이크로소프트 에이전트 보안을 위한 가시성 확보 강조, AI 16
기술･연구
∙앤트로픽 연구 결과 개발자의 사용은 실력 향상에는 역효과, AI 18
∙네이처 과학 출판에서 책임 있는 활용 가이드라인 제시, AI 19
∙앤트로픽 책임 있는 확장 정책을 버전으로 업데이트, 3.0 20
인력･교육
∙ 채용의 부정적 영향과 개선 방안AI 22
시대 근로자의 판단력 향상을 위해 업무 방식의 재설계 필요AI ∙ 23
확산에 직면해 기술 현장직으로 전환하는 사무직 근로자 증가AI ·∙ 24
로버트 라이시 미국 노동부 장관 로 인한 불평등 확산 경고, AI前 ∙ 25
주요행사일정∙ 년 국내외 인공지능 주요 행사 2026 26
Ⅰ 년 월호2026 3 Ⅰ


In [ ]:
# RecursiveCharacterTextSplitter: 문서를 의미 단위로 분할합니다.
# chunk_size: 각 청크의 최대 문자 수 (1000자)
# chunk_overlap: 인접 청크 간 겹치는 문자 수 (200자) — 경계 부분의 정보 손실 방지
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200,
)
splits = text_splitter.split_documents(docs)

# 분할 결과 확인: 각 청크의 앞부분을 출력합니다.
for i, doc in enumerate(splits):
    print(f"청크 {i}: {doc.page_content[:60]}...")
print(f"총 청크 수: {len(splits)}")

청크 0: LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크입니다....
청크 1: 에이전트는 도구를 사용하여 외부 시스템과 상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다....
총 청크 수: 2


## 5.4 벡터 스토어 구축

벡터 스토어는 임베딩을 인덱싱하고 유사도 검색을 수행하는 전문 데이터베이스입니다.
`InMemoryVectorStore`는 별도 인프라 없이 사용할 수 있어 개발/테스트용으로 적합합니다.

### 주요 벡터 스토어 비교

| 벡터 스토어 | 유형 | 적합한 경우 |
|---|---|---|
| `InMemoryVectorStore` | 인프로세스 | 개발, 소규모 데이터셋 |
| `Chroma` | 임베디드/클라이언트-서버 | 프로토타이핑, 중규모 데이터셋 |
| `FAISS` | 인프로세스 | 고성능 로컬 검색 |
| `Pinecone` | 매니지드 클라우드 | 프로덕션, 확장성 |
| `PGVector` | PostgreSQL 확장 | 기존 PostgreSQL 인프라 활용 |

### 검색 유형

| 검색 타입 | 설명 |
|---|---|
| `"similarity"` | 표준 최근접 이웃 검색 — 코사인 유사도 기반 |
| `"mmr"` | Maximal Marginal Relevance — 관련성과 다양성의 균형 (중복 감소) |

In [15]:
from langchain_core.vectorstores import InMemoryVectorStore

# from_documents: 문서 청크를 임베딩으로 변환하여 벡터 스토어에 저장합니다.
# 내부적으로 embeddings.embed_documents()를 호출하여 각 청크를 벡터화합니다.
vector_store = InMemoryVectorStore.from_documents(
    documents=splits, embedding=embeddings,
)

# similarity_search: 쿼리와 가장 유사한 상위 k개 문서를 반환합니다.
test_results = vector_store.similarity_search("LangGraph", k=2)
for doc in test_results:
    print(f"  [{doc.metadata['source']}] {doc.page_content[:80]}")
print(f"벡터 스토어 준비 완료. 문서 {len(splits)}개.")

  [langgraph-docs] LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크입니다.
  [agent-guide] 에이전트는 도구를 사용하여 외부 시스템과 상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다.
벡터 스토어 준비 완료. 문서 2개.


## 5.5 검색 도구 정의

`@tool` 데코레이터로 벡터 검색을 LangChain 도구로 등록합니다.

`response_format="content_and_artifact"`를 사용하면 도구 출력을 두 부분으로 분리합니다:
- **Content**: 모델에 전달되는 문자열 표현 (추론에 사용)
- **Artifact**: 원본 Document 객체 (프로그래밍 방식으로 접근 가능하지만 모델에 전송되지 않음)

이 분리를 통해 모델에는 읽기 쉬운 텍스트를, 후속 처리에는 메타데이터가 포함된 원본 객체를 사용할 수 있습니다.

In [16]:
from langchain_core.tools import tool


# response_format="content_and_artifact": 모델용 텍스트와 프로그래밍용 원본 객체를 분리합니다.
@tool(response_format="content_and_artifact")
def retrieve(query: str):
    """지식 베이스에서 관련 문서를 검색합니다."""
    # 쿼리와 가장 유사한 상위 4개 문서를 벡터 검색합니다.
    docs = vector_store.similarity_search(query, k=4)
    # Content: 출처와 본문을 포맷팅한 문자열 (모델이 읽을 텍스트)
    serialized = "\n\n".join(
        f"출처: {d.metadata.get('source', '?')}\n{d.page_content}"
        for d in docs
    )
    # (Content, Artifact) 튜플 반환 — Artifact는 원본 Document 리스트
    return serialized, docs

## 5.6 LangChain RAG Agent -- `create_agent` + `@tool`

가장 간단한 방법: retriever를 도구로 등록하고 에이전트가 **필요할 때 자율적으로** 호출합니다.

### 다중 단계 검색 흐름
RAG Agent는 자동으로 다중 검색 단계를 실행할 수 있습니다:
1. **초기 검색** — 사용자 질문 기반 쿼리 생성
2. **결과 평가** — 검색된 문서가 질문에 충분한지 판단
3. **재구성 및 재검색** — 결과가 부족하면 쿼리를 수정하여 재검색
4. **통합** — 모든 검색 결과를 결합하여 최종 답변 생성

> 💡 이 접근법은 복잡한 리서치 질문에 적합하지만, 여러 번의 LLM 호출로 비용과 지연이 증가합니다.

In [17]:
from langchain.agents import create_agent

# retrieve 도구를 바인딩한 RAG 에이전트를 생성합니다.
# 에이전트가 질문을 받으면 자율적으로 retrieve 도구 호출 여부를 결정합니다.
rag_agent = create_agent(
    model=llm, tools=[retrieve],
    system_prompt="당신은 리서치 어시스턴트입니다. "
    "답변하기 전에 retrieve를 사용하여 검색하세요.",
)

# 에이전트 실행: LLM이 retrieve 도구를 호출 → 검색 결과를 바탕으로 답변 생성
response = rag_agent.invoke(
    {"messages": [{"role": "user", "content": "LangGraph란 무엇인가요?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)

LangGraph는 **LLM(대규모 언어 모델) 기반의 상태 기반 멀티 에이전트 애플리케이션**을 만들기 위한 프레임워크입니다.

쉽게 말하면:

- **대화/작업의 상태(state)** 를 유지하면서
- **여러 에이전트나 노드(node)** 가
- **조건 분기, 반복, 도구 호출** 등을 거치며
- 복잡한 흐름을 안정적으로 구성할 수 있게 해줍니다.

### 왜 쓰나요?
일반적인 LLM 앱은 한 번의 프롬프트-응답에는 강하지만,  
여러 단계의 작업이나 다음 행동을 제어해야 하는 경우가 어렵습니다.  
LangGraph는 이런 상황에서 유용합니다.

예를 들면:
- 고객 문의를 분류한 뒤 적절한 에이전트로 넘기기
- 검색 → 요약 → 검증 → 최종 응답 같은 다단계 처리
- 실패하면 재시도하거나 다른 경로로 분기하기
- 여러 도구를 사용하는 에이전트 워크플로우 만들기

### 핵심 특징
- **상태 관리**: 대화나 작업의 진행 상태를 저장
- **그래프 구조**: 노드와 엣지로 흐름을 구성
- **조건부 분기**: 상황에 따라 다음 단계 결정
- **반복/순환 지원**: 필요한 만큼 반복 가능
- **에이전트 오케스트레이션**: 여러 에이전트를 협업시키기 좋음

### 한 줄 요약
**LangGraph는 LLM 앱을 “흐름 제어가 가능한 상태 머신/그래프” 형태로 만들게 해주는 프레임워크입니다.**

원하시면 제가 이어서  
1) **LangChain과의 차이**, 또는  
2) **간단한 예제 코드**  
로도 설명해드릴게요.


## 5.7 LangChain RAG Chain -- `@dynamic_prompt` 미들웨어

**단일 LLM 호출**로 RAG를 구현합니다. `@dynamic_prompt`가 LLM 호출 전에 문서를 검색하고 시스템 프롬프트에 자동으로 주입합니다. 미들웨어 방식이므로 에이전트 루프 없이 **단일 패스**로 동작합니다.

| 특성 | RAG Agent | RAG Chain |
|---|---|---|
| LLM 호출 수 | 다중 (에이전트 결정) | 단일 |
| 검색 횟수 | 1회 이상 (에이전트 제어) | 정확히 1회 (미들웨어 제어) |
| 쿼리 재구성 | 자동 | 미지원 |
| 지연 | 높음 (여러 왕복) | 낮음 (단일 패스) |
| 비용 | 높음 (더 많은 토큰) | 낮음 (더 적은 토큰) |
| 투명성 | 에이전트 추론이 메시지에 노출 | 컨텍스트 주입이 암묵적 |

> 💡 **고급 활용**: `@dynamic_prompt`로 기본 컨텍스트를 주입하면서 동시에 retriever 도구를 제공하여 두 접근법을 결합할 수도 있습니다.

In [18]:
from langchain.agents.middleware import dynamic_prompt


# @dynamic_prompt: LLM 호출 전에 실행되는 미들웨어입니다.
# 사용자 메시지를 기반으로 벡터 검색을 수행하고,
# 검색 결과를 시스템 프롬프트에 자동 주입합니다.
@dynamic_prompt
def rag_prompt(request):
    """문서를 검색하여 시스템 프롬프트에 주입합니다."""
    # 가장 최근 사용자 메시지에서 검색 쿼리를 추출합니다.
    user_msg = request.state["messages"][-1].content
    # 벡터 스토어에서 유사 문서 4개를 검색합니다.
    docs = vector_store.similarity_search(user_msg, k=4)
    # 검색된 문서를 하나의 컨텍스트 문자열로 결합합니다.
    ctx = "\n\n".join(d.page_content for d in docs)
    # 반환값이 시스템 프롬프트로 사용됩니다.
    return f"컨텍스트를 기반으로 답변하세요:\n\n{ctx}"

In [19]:
# rag_prompt 미들웨어를 적용한 에이전트 생성
# 도구 없이 미들웨어만 사용 → 단일 LLM 호출로 RAG 수행
chain = create_agent(model=llm, middleware=[rag_prompt])

# 질문 시 @dynamic_prompt가 자동으로 관련 문서를 검색하여 프롬프트에 주입합니다.
resp = chain.invoke(
    {"messages": [{"role": "user", "content": "에이전트는 어떻게 작동하나요?"}]},
    config=lf_config,
)
print(resp["messages"][-1].content)

에이전트는 **목표를 달성하기 위해 스스로 다음 행동을 결정하며** 움직이는 시스템입니다.  
질문에 답만 하는 게 아니라, 필요하면 **도구를 사용하고**, 그 결과를 보고 다시 판단합니다.

핵심 흐름은 보통 이렇게 됩니다:

1. **상황 파악**  
   현재 입력과 상태를 봅니다.

2. **추론**  
   지금 무엇이 필요한지 생각합니다.  
   예: 검색이 필요한가? 계산이 필요한가? 다른 시스템을 호출해야 하나?

3. **행동(Action)**  
   도구를 사용하거나 외부 시스템과 상호작용합니다.  
   예: API 호출, 검색, DB 조회 등.

4. **관찰(Observation)**  
   행동 결과를 확인합니다.

5. **반복**  
   결과가 충분하면 답을 만들고, 아니면 다음 행동을 다시 선택합니다.

이 방식이 바로 **ReAct 패턴**과 잘 맞습니다.  
즉, **Reasoning(추론)과 Acting(행동)** 을 번갈아 수행합니다.

또한 **LangGraph**는 이런 에이전트/멀티 액터 시스템을 **상태(state) 기반**으로 구성하는 데 유용한 프레임워크입니다.  
여러 에이전트가 각자 역할을 나누고, 상태를 공유하며, 조건에 따라 다음 단계로 흐름을 제어할 수 있습니다.

원하시면 제가  
- **에이전트 작동 방식 예시**,  
- **ReAct와 일반 LLM의 차이**,  
- **LangGraph로 에이전트를 구성하는 구조**  
중 하나를 더 자세히 설명해드릴게요.


## 5.8 LangGraph 커스텀 RAG -- StateGraph 구축

LangGraph `StateGraph`로 세밀한 제어가 가능한 RAG 에이전트를 직접 구축합니다.

이 방식의 핵심 장점은 **조건부 라우팅**을 통해:
- 검색 결과의 **관련성을 LLM이 평가** (GradeDocuments 구조화 출력)
- 관련 없는 경우 **쿼리를 리라이트하여 재검색**
- 각 단계를 **노드와 엣지로 명시적으로 정의**하여 디버깅과 모니터링이 용이

### 아키텍처

```
        [generate_query_or_respond]      ← 진입 노드: 검색할지 직접 답변할지 결정
             /              \
       (tool call)       (no tool call)
           |                  |
      [retrieve]           [END]         ← 도구 호출 없으면 바로 종료
           |
   [grade_documents]                     ← 검색 결과의 관련성 평가
      /          \
(relevant)    (not relevant)
    |              |
[generate]   [rewrite_question]          ← 관련 없으면 쿼리 재작성 후 재시도
    |              |
  [END]    [generate_query_or_respond]
```

In [20]:
from langgraph.graph import MessagesState


class AgentState(MessagesState):
    """커스텀 RAG 에이전트 상태.
    
    MessagesState를 상속하여 messages 필드를 기본 포함하고,
    relevance 필드를 추가하여 문서 관련성 평가 결과를 저장합니다.
    """
    relevance: str  # "relevant" 또는 "not_relevant" — grade_documents 노드가 설정


print(f"AgentState 키: {list(AgentState.__annotations__)}")

AgentState 키: ['messages', 'relevance']


## 5.9 `generate_query_or_respond` 노드

그래프의 **진입 노드**입니다. LLM이 retrieve 도구를 호출할지, 직접 응답할지 결정합니다.

- 지식 베이스 관련 질문 → `retrieve` 도구 호출 (tool call 메시지 생성)
- 일반 대화/인사 → 도구 호출 없이 직접 응답

In [21]:
# retrieve 도구를 LLM에 바인딩합니다.
# bind_tools를 사용하면 LLM이 도구 호출 여부를 자율적으로 결정합니다.
llm_with_tools = llm.bind_tools([retrieve])


def generate_query_or_respond(state: AgentState):
    """검색할지 직접 응답할지 결정합니다.
    
    LLM이 도구 호출이 필요하다고 판단하면 tool_calls가 포함된 AIMessage를,
    그렇지 않으면 일반 텍스트 AIMessage를 반환합니다.
    """
    system = ("당신은 유용한 어시스턴트입니다. "
              "지식 베이스 관련 질문에는 retrieve 도구를 사용하세요.")
    msgs = [{"role": "system", "content": system}] + state["messages"]
    response = llm_with_tools.invoke(msgs, config=lf_config)
    return {"messages": [response]}

## 5.10 `grade_documents` 노드 -- 구조화 출력으로 관련성 평가

`GradeDocuments` Pydantic 스키마로 LLM이 문서 관련성을 **이진 분류**(relevant/not_relevant)합니다.
`with_structured_output`을 사용하면 LLM 응답이 자동으로 파싱되어 타입 안전한 객체로 반환됩니다.

이 평가 결과에 따라 다음 노드가 결정됩니다:
- `relevant` → `generate_answer` 노드로 이동
- `not_relevant` → `rewrite_question` 노드로 이동하여 쿼리를 재작성

In [12]:
from pydantic import BaseModel, Field
from typing import Literal


class GradeDocuments(BaseModel):
    """검색된 문서의 이진 관련성 점수.
    
    LLM이 이 스키마에 맞춰 구조화된 응답을 생성합니다.
    Literal 타입으로 출력 값을 제한하여 예측 가능한 라우팅을 보장합니다.
    """
    relevance: Literal["relevant", "not_relevant"] = Field(
        description="문서가 질문에 관련이 있으면 'relevant', 없으면 'not_relevant'"
    )
    reasoning: str = Field(description="판단 이유에 대한 간략한 설명")


# with_structured_output: LLM 응답을 GradeDocuments 객체로 자동 파싱합니다.
grader = llm.with_structured_output(GradeDocuments)

In [13]:
def grade_documents(state: AgentState):
    """
    검색된 문서의 관련성을 평가합니다.
    """

    msgs = state["messages"]

    user_q = next(
        (m.content for m in msgs if m.type == "human"),
        ""
    )

    tool_content = msgs[-1].content

    grade = grader.invoke(
        f"질문: {user_q}\n문서:\n{tool_content}\n"
        f"이 문서들이 관련이 있습니까?"
    )

    return {
        "relevance": grade.relevance,
        "messages": msgs
    }

## 5.11 `rewrite_question` 노드

검색된 문서가 **관련 없다고 평가**되었을 때 실행됩니다.
원래 질문을 더 구체적인 용어로 리라이트하여 벡터 검색의 품질을 향상시킵니다.

리라이트된 질문은 `generate_query_or_respond` 노드로 돌아가 재검색을 시도합니다.

In [14]:
def rewrite_question(state: AgentState):
    """
    검색 품질 향상을 위해 질문을 리라이트합니다.
    """

    user_q = next(
        (m.content for m in state["messages"] if m.type == "human"),
        ""
    )

    prompt = (
        f"다른 용어를 사용하여 리라이트하세요.\n"
        f"원본: {user_q}\n"
        f"리라이트:"
    )

    response = llm.invoke(prompt, config=lf_config)

    return {
        "messages": [
            {
                "role": "human",
                "content": response.content
            }
        ]
    }

## 5.12 `generate_answer` 노드

관련 문서가 확인되면 실행되는 **최종 답변 생성 노드**입니다.
검색된 문서(컨텍스트)와 원본 질문을 결합하여 LLM이 최종 답변을 생성합니다.

> 💡 프롬프트에서 "컨텍스트만을 사용하여 답변하세요"라고 명시하여 LLM이 검색 결과에 기반한 답변만 생성하도록 제한합니다 (할루시네이션 방지).

In [15]:
def generate_answer(state: AgentState):
    """
    검색된 문서를 사용하여 답변을 생성합니다.
    """

    user_q = next(
        (m.content for m in state["messages"] if m.type == "human"),
        ""
    )

    docs = next(
        (m.content for m in state["messages"] if m.type == "tool"),
        ""
    )

    prompt = (
        f"컨텍스트만을 사용하여 답변하세요.\n"
        f"컨텍스트:\n{docs}\n\n"
        f"질문: {user_q}"
    )

    resp = llm.invoke(prompt, config=lf_config)

    return {
        "messages": [
            {
                "role": "assistant",
                "content": resp.content
            }
        ]
    }

## 5.13 그래프 조립 & 실행

모든 노드를 `StateGraph`에 등록하고, 조건부 엣지로 연결합니다.

- `tools_condition`: LLM 응답에 tool_calls가 있으면 `"tools"`, 없으면 `"__end__"` 반환
- `relevance_router`: `state["relevance"]` 값에 따라 `generate_answer` 또는 `rewrite_question`으로 분기

아래 3개 셀에서 순서대로 노드 등록 → 엣지 연결 → 컴파일 & 실행을 수행합니다.

In [16]:
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


def relevance_router(state: AgentState):
    """문서 관련성에 따라 다음 노드를 결정하는 라우터 함수."""
    if state.get("relevance") == "relevant":
        return "generate_answer"    # 관련 문서 → 답변 생성
    return "rewrite_question"       # 비관련 문서 → 쿼리 재작성


# StateGraph 생성 — AgentState를 상태 스키마로 사용
graph = StateGraph(AgentState)

# 노드 등록: 각 함수를 고유 이름의 노드로 등록합니다.
graph.add_node("gen_query", generate_query_or_respond)

In [17]:
# ToolNode: retrieve 도구의 실행을 담당하는 내장 노드
graph.add_node("retrieve", ToolNode([retrieve]))
graph.add_node("grade_documents", grade_documents)
graph.add_node("rewrite_question", rewrite_question)
graph.add_node("generate_answer", generate_answer)

# 엣지 연결: START → gen_query (진입점)
graph.add_edge(START, "gen_query")

# 조건부 엣지: gen_query의 LLM 응답에 tool_calls가 있으면 retrieve로,
# 없으면 END로 이동합니다. tools_condition은 LangGraph 내장 함수입니다.
graph.add_conditional_edges(
    "gen_query", tools_condition,
    {"tools": "retrieve", "__end__": END},
)

In [18]:
# retrieve 완료 후 항상 grade_documents로 이동 (무조건 엣지)
graph.add_edge("retrieve", "grade_documents")

# 조건부 엣지: 문서 관련성 평가 결과에 따라 분기
# relevant → generate_answer, not_relevant → rewrite_question
graph.add_conditional_edges(
    "grade_documents", relevance_router,
    {"generate_answer": "generate_answer",
     "rewrite_question": "rewrite_question"},
)

# rewrite_question 후 gen_query로 돌아가 재검색 시도 (루프)
graph.add_edge("rewrite_question", "gen_query")
# generate_answer 완료 후 종료
graph.add_edge("generate_answer", END)

# 그래프 컴파일: 모든 노드와 엣지가 유효한지 검증 후 실행 가능한 앱 생성
app = graph.compile()
print("그래프 컴파일 성공.")

그래프 컴파일 성공.


In [19]:
# 그래프 스트리밍 실행: 각 노드의 출력을 실시간으로 확인합니다.
# stream()은 각 노드가 완료될 때마다 {노드명: 출력} 딕셔너리를 yield합니다.
for event in app.stream(
    {"messages": [{"role": "user", "content": "LangGraph란 무엇인가요?"}]},
    config=lf_config,
):
    for node_name, output in event.items():
        print(f"--- {node_name} ---")
        if "messages" in output:
            last = output["messages"][-1]
            txt = last.content if hasattr(last, "content") else str(last)
            print(txt[:200])

--- gen_query ---

--- retrieve ---
출처: langgraph-docs
LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크입니다.

출처: agent-guide
에이전트는 도구를 사용하여 외부 시스템과 상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다.


D:\deepagents\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=GradeDocuments(relevance=...성이 있습니다.'), input_type=GradeDocuments])
  return self.__pydantic_serializer__.to_python(


--- grade_documents ---
출처: langgraph-docs
LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크입니다.

출처: agent-guide
에이전트는 도구를 사용하여 외부 시스템과 상호작용합니다. ReAct 패턴은 추론과 행동을 번갈아 수행합니다.


--- generate_answer ---
{'role': 'assistant', 'content': 'LangGraph는 LLM으로 상태 기반 멀티 액터 애플리케이션을 구축하기 위한 프레임워크입니다.'}


## 요약

### 세 가지 RAG 접근법 비교

| 특성 | RAG Agent | RAG Chain | LangGraph 커스텀 |
|---|---|---|---|
| LLM 호출 수 | 다중 | 단일 | 다중 |
| 검색 횟수 | 에이전트 결정 | 정확히 1회 | 커스텀 |
| 쿼리 재구성 | 자동 | 미지원 | 명시적 노드 |
| 관련성 평가 | 암묵적 | 없음 | `GradeDocuments` |
| 제어 수준 | 낮음 | 낮음 | 높음 |
| 구현 복잡도 | 낮음 | 최저 | 높음 |

### 선택 가이드
- **빠른 프로토타이핑** → RAG Chain (`@dynamic_prompt`)
- **자율적 다중 검색이 필요한 경우** → RAG Agent (`create_agent` + `@tool`)
- **관련성 평가, 쿼리 리라이트 등 세밀한 제어** → LangGraph 커스텀 RAG

### 핵심 LangGraph 패턴

| 패턴 | 구현 |
|---|---|
| 조건부 라우팅 | `add_conditional_edges` + `tools_condition` |
| 구조화 출력 | `llm.with_structured_output(GradeDocuments)` |
| 도구 노드 | `ToolNode([retrieve])` |